# ANN for numeric video features

A compact multilayer perceptron predicts `final_label` (0/1) from views, likes, duration in seconds, and the like ratio (likes / views). The notebook uses the existing stratified train, validation, and test splits. It transforms skewed counts, fits scaling on training data only, applies L2 regularization, and uses validation loss with patience-based early stopping. The test split is evaluated once after model selection.

## 1. Imports and reproducibility

In [ ]:
from copy import deepcopy
from pathlib import Path
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


## 2. Load the prepared splits

Run from the repository root or from `models/`. No new split is created, so the held-out test set remains untouched during training and tuning.

In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed').exists() and (ROOT.parent / 'data' / 'processed').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data' / 'processed'
train_df = pd.read_csv(DATA_DIR / 'train.csv')
validation_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

TARGET = 'final_label'
REQUIRED_COLUMNS = ['views', 'likes', 'duration', TARGET]
for split_name, split_df in [('train', train_df), ('validation', validation_df), ('test', test_df)]:
    missing = [column for column in REQUIRED_COLUMNS if column not in split_df.columns]
    if missing:
        raise KeyError(f'{split_name} split is missing columns: {missing}')
    print(f'{split_name}: {len(split_df):,} rows')

train_df[TARGET].value_counts(normalize=True).sort_index().rename('train_fraction')


## 3. Prepare numeric features

`duration` is parsed as seconds (accepting values such as `1954s` as well as numeric values). `like_ratio` is computed safely and is zero when views are zero. `log1p` reduces the extreme skew in views and likes. Rows with missing or invalid values in required fields are excluded consistently per split.

In [ ]:
FEATURES = ['views', 'likes', 'duration_seconds', 'like_ratio']

def duration_to_seconds(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    text = str(value).strip().lower()
    match = re.fullmatch(r'(\d+(?:\.\d+)?)\s*s?', text)
    return float(match.group(1)) if match else np.nan

def prepare_split(frame):
    data = frame.copy()
    data['views'] = pd.to_numeric(data['views'], errors='coerce')
    data['likes'] = pd.to_numeric(data['likes'], errors='coerce')
    data['duration_seconds'] = data['duration'].map(duration_to_seconds)
    data[TARGET] = pd.to_numeric(data[TARGET], errors='coerce')
    data['like_ratio'] = data['likes'].div(data['views'].replace(0, np.nan)).fillna(0.0)
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna(subset=['views', 'likes', 'duration_seconds', TARGET]).copy()
    if (data[['views', 'likes', 'duration_seconds']] < 0).any().any():
        raise ValueError('Views, likes, and duration must be non-negative for log1p.')
    data['views'] = np.log1p(data['views'])
    data['likes'] = np.log1p(data['likes'])
    data['duration_seconds'] = np.log1p(data['duration_seconds'])
    data[TARGET] = data[TARGET].astype(int)
    if not set(data[TARGET].unique()).issubset({0, 1}):
        raise ValueError(f'{TARGET} must contain only binary labels 0 and 1.')
    return data

train = prepare_split(train_df)
validation = prepare_split(validation_df)
test = prepare_split(test_df)

X_train_raw, y_train = train[FEATURES], train[TARGET].to_numpy()
X_val_raw, y_val = validation[FEATURES], validation[TARGET].to_numpy()
X_test_raw, y_test = test[FEATURES], test[TARGET].to_numpy()

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

print('Rows after numeric validation:', {'train': len(train), 'validation': len(validation), 'test': len(test)})
print('Feature order:', FEATURES)


## 4. Train with validation-based early stopping

This deliberately small network has one 16-unit hidden layer. L2 weight decay (`alpha`) discourages overly complex fits. After each epoch the model is scored on the validation set; the best weights are saved and training stops when validation log loss has not improved for 20 epochs. The test data is not used here.

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(16,),
    activation='relu',
    solver='adam',
    alpha=0.01,
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=1,
    warm_start=True,
    shuffle=True,
    random_state=RANDOM_STATE,
)

MAX_EPOCHS = 300
PATIENCE = 20
MIN_DELTA = 1e-4
classes = np.array([0, 1])
history = {'train_loss': [], 'validation_loss': []}
best_validation_loss = np.inf
best_model = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    if epoch == 1:
        model.partial_fit(X_train, y_train, classes=classes)
    else:
        model.partial_fit(X_train, y_train)

    train_probabilities = model.predict_proba(X_train)
    validation_probabilities = model.predict_proba(X_val)
    train_loss = log_loss(y_train, train_probabilities, labels=classes)
    validation_loss = log_loss(y_val, validation_probabilities, labels=classes)
    history['train_loss'].append(train_loss)
    history['validation_loss'].append(validation_loss)

    if validation_loss < best_validation_loss - MIN_DELTA:
        best_validation_loss = validation_loss
        best_model = deepcopy(model)
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f'Epoch {epoch:03d} | train log loss {train_loss:.4f} | validation log loss {validation_loss:.4f}')

    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch}; best validation epoch was {best_epoch}.')
        break

model = best_model
print(f'Using epoch {best_epoch} (best validation log loss: {best_validation_loss:.4f})')


In [ ]:
epochs = np.arange(1, len(history['train_loss']) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history['train_loss'], label='Training')
plt.plot(epochs, history['validation_loss'], label='Validation')
plt.axvline(best_epoch, color='gray', linestyle='--', label=f'Best epoch ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('Log loss')
plt.title('ANN learning curves')
plt.legend()
plt.tight_layout()
plt.show()


## 5. Check generalization on validation data

The validation report is for model selection and a generalization check. A large gap between training and validation scores can indicate overfitting.

In [ ]:
def evaluate_split(name, fitted_model, features, labels):
    probabilities = fitted_model.predict_proba(features)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    metrics = {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0),
        'roc_auc': roc_auc_score(labels, probabilities) if len(np.unique(labels)) == 2 else np.nan,
        'log_loss': log_loss(labels, fitted_model.predict_proba(features), labels=[0, 1]),
    }
    print(f'--- {name} ---')
    print(pd.Series(metrics).round(4))
    print('Confusion matrix [ [TN, FP], [FN, TP] ]:')
    print(confusion_matrix(labels, predictions, labels=[0, 1]))
    print(classification_report(labels, predictions, labels=[0, 1], zero_division=0))
    return metrics

train_metrics = evaluate_split('Train', model, X_train, y_train)
validation_metrics = evaluate_split('Validation', model, X_val, y_val)
print('Train minus validation F1 gap:', round(train_metrics['f1'] - validation_metrics['f1'], 4))


## 6. Final held-out test evaluation

Run this section after training and validation checks. These are the final generalization metrics on the untouched test split.

In [ ]:
test_metrics = evaluate_split('Test (held out)', model, X_test, y_test)


## Notes

This is a binary classifier: it predicts `final_label` from the four numeric features. The like ratio uses the available `likes` and `views` counts. If a different numeric target is intended, change `TARGET` and the output/loss setup accordingly; this notebook currently follows the repository's clickbait label task.